# Cart-pole — LQR on an infinite horizon, on a finite horizon, and along a trajectory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/optimal_control/cartpole_lqr.ipynb)

Balancing a pole on a cart is the standard test of **linear-quadratic regulation**: the plant is nonlinear, but near the upright equilibrium a linear model is accurate, and a quadratic cost makes the optimal state feedback computable in closed form. This notebook builds that feedback three times for the **same** plant and the **same** weights $Q$, $R$:

1. **Infinite horizon** — the algebraic Riccati equation gives one constant gain $K$ that balances the pole.
2. **Finite horizon** — the Riccati *differential* equation, integrated backward from a terminal weight $S_f$, gives a gain schedule $K(t)$.
3. **Along a trajectory** — a swing-up from hanging is planned by direct collocation, the plant is linearized along it, and the same finite-horizon design gives the feedback $u = u_d(t) - K(t)\,(x - x_d(t))$ that keeps the real cart-pole on the plan.

Each design is run on the full nonlinear cart-pole. The last section writes the backward Riccati recursion by hand, with an Euler step, to see what an approximate integration can do to $S(t)$.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox. The LQR design lives in [`control/lqr.py`](https://github.com/alx87grd/minilink/blob/main/minilink/control/lqr.py); the library workflow is in [`showcase_minilink`](../../tutorial/showcase_minilink.ipynb) and [`03_control`](../../tutorial/03_control.ipynb).

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import importlib.util

import numpy as np

from minilink import (
    CartPole,
    PlanningProblem,
    QuadraticCost,
    TrajectoryOptimizationPlanner,
    TrajectorySource,
    linearize,
    lqr,
    lqr_finite_horizon,
    trajectory_lqr,
)
from minilink.control.lqr import lqr_gain_schedule

## 1. Plant and operating point

The catalog `CartPole` has state $x = [x_c,\;\theta,\;\dot x_c,\;\dot\theta]$ (cart position, pole angle, and their rates) and one input, the horizontal force $u$ on the cart. The pole hangs down at $\theta = 0$; the balance point is the upright equilibrium
$$\bar x = [0,\;\pi,\;0,\;0], \qquad \bar u = 0 .$$
The weights below penalize the angle ten times more than the cart position.

In [ ]:
X_BAR = np.array([0.0, np.pi, 0.0, 0.0])  # cart at the origin, pole upright
U_BAR = np.array([0.0])
Q = np.diag([1.0, 10.0, 1.0, 1.0])
R = np.array([[1.0]])
TF = 10.0

plant = CartPole()

## 2. Linearized model

Around $(\bar x, \bar u)$ the dynamics $\dot x = f(x, u)$ reduce to
$$\dot{\tilde x} = A\,\tilde x + B\,\tilde u, \qquad \tilde x = x - \bar x,\quad \tilde u = u - \bar u,$$
with $A = \partial f/\partial x$ and $B = \partial f/\partial u$ evaluated at the operating point. `linearize` returns this linear system; one eigenvalue of $A$ is positive — the upright pole falls on its own.

In [ ]:
lti = linearize(plant, x_bar=X_BAR, u_bar=U_BAR)
A, B = lti.A(), lti.B()

print("A =\n", np.round(A, 3))
print("B =\n", np.round(B, 3))
print("open-loop poles:", np.round(np.linalg.eigvals(A), 3))

## 3. Infinite horizon: the algebraic Riccati equation

The cost to minimize is
$$J = \int_0^{\infty} \big( \tilde x^T Q\, \tilde x + \tilde u^T R\, \tilde u \big)\, dt .$$
Its optimal cost-to-go is quadratic, $J^*(\tilde x) = \tilde x^T S\, \tilde x$, where $S$ solves the **algebraic Riccati equation**
$$0 = S A + A^T S - S B R^{-1} B^T S + Q ,$$
and the optimal policy is the constant state feedback
$$u = \bar u - K\,(x - \bar x), \qquad K = R^{-1} B^T S .$$
`lqr` solves the equation and returns the feedback block, ready to wire.

In [ ]:
ctl_inf = lqr(A, B, Q, R, xbar=X_BAR, ubar=U_BAR)
K_inf = ctl_inf.params["K"]

print("K =", np.round(K_inf, 3))
print("closed-loop poles:", np.round(np.linalg.eigvals(A - B @ K_inf), 3))

## 4. Closed loop from several initial states

The gain was designed on the linear model; the simulation runs the **nonlinear** cart-pole. `controller @ plant` wires the state feedback $u = \pi(x)$. All three starts below are caught, but the force the linear law asks for grows with the distance from $\bar x$: about 9 N from the first start, 31 N from the third. Push the tilt further (section 7) and the linear model stops describing the plant the law is acting on.

In [ ]:
def simulate(controller, x0, tf=TF):
    """Close the loop on a fresh cart-pole and simulate from x0."""
    plant = CartPole()
    plant.x0 = np.array(x0)
    loop = controller @ plant  # state feedback: plant.x -> ctl.x, ctl.u -> plant.u
    traj = loop.compute_trajectory(tf=tf, verbose=False)
    return loop, plant, traj


X0_LIST = [
    [0.5, np.pi + 0.3, 0.0, 0.0],  # cart displaced, pole tilted 17 degrees
    [-1.0, np.pi - 0.6, 0.0, 0.0],  # the other way, 34 degrees
    [0.0, np.pi + 1.0, 0.0, 0.0],  # 57 degrees: far from the linear model
]

for x0 in X0_LIST:
    loop, plant, traj = simulate(ctl_inf, x0)
    loop.plot_trajectory(traj, signals=("x", "u"))

Animation of the last run, under the constant gain.

In [ ]:
loop.animate(traj)

## 5. Finite horizon: the Riccati differential equation

On a horizon $t_f$ with a terminal weight $S_f$,
$$J = \int_0^{t_f} \big( \tilde x^T Q\, \tilde x + \tilde u^T R\, \tilde u \big)\, dt + \tilde x(t_f)^T S_f\, \tilde x(t_f),$$
the cost-to-go is still quadratic, $J^*(\tilde x, t) = \tilde x^T S(t)\, \tilde x$, but $S$ now depends on time. Dynamic programming gives it as the solution of the **Riccati differential equation**, integrated *backward* from the terminal condition:
$$-\dot S = S A + A^T S - S B R^{-1} B^T S + Q, \qquad S(t_f) = S_f ,$$
and the optimal feedback becomes a **gain schedule**
$$u = \bar u - K(t)\,(x - \bar x), \qquad K(t) = R^{-1} B^T S(t) .$$
`lqr_finite_horizon` solves the equation exactly on a time grid: with the co-state $\lambda = S x$ of the maximum principle, the pair $(x, \lambda)$ follows a *linear* Hamiltonian system, so one step backward in time is one matrix exponential applied through a linear-fractional map — no integration error, no stiffness, and $S$ symmetric by construction. It returns the schedule as a feedback block; `plot_gain_schedule` draws $K(t)$. Far from $t_f$ the schedule settles on the constant gain of section 3; near $t_f$ it follows $S_f$. With $S_f = 0$ the gains vanish at the end: there is no reason to act when no cost remains.

In [ ]:
S_F = np.zeros((4, 4))  # no terminal penalty

ctl_fin = lqr_finite_horizon(A, B, Q, R, S_F, TF, xbar=X_BAR, ubar=U_BAR)
ctl_fin.plot_gain_schedule()

The block applies $K(t)$ at the simulation time, interpolated between samples. Past $t_f$ it keeps $K(t_f)$ by default, or switches to the stationary gain with `after=\"stationary\"`. Same initial states as above, same plant.

In [ ]:
for x0 in X0_LIST:
    loop, plant, traj = simulate(ctl_fin, x0)
    loop.plot_trajectory(traj, signals=("x", "u"))

### The two designs scored on the same cost

Both loops are scored with the same quadratic $J$ over $[0, t_f]$, evaluated on the nonlinear plant's trajectory. The finite-horizon design minimizes exactly this integral for the linear model, so it cannot lose to the constant gain where the linear model holds. Here the two costs agree to three decimals: with $S_f = 0$ the schedule equals the constant gain over the whole stretch where the state is still moving, and only departs from it in the last seconds, when nothing is left to regulate. A terminal weight or a short horizon separates the two designs (section 7).

In [ ]:
cost = QuadraticCost.from_system(plant, xbar=X_BAR, ubar=U_BAR, Q=Q, R=R)

print("x0                          J infinite   J finite")
for x0 in X0_LIST:
    loop_inf, plant_inf, traj_inf = simulate(ctl_inf, x0)
    loop_fin, plant_fin, traj_fin = simulate(ctl_fin, x0)
    J_inf = loop_inf.compute_cost(cost, of=plant_inf, traj=traj_inf)
    J_fin = loop_fin.compute_cost(cost, of=plant_fin, traj=traj_fin)
    print(f"{np.round(x0, 2)!s:<28}{J_inf:10.3f} {J_fin:10.3f}")

## 6. Along a trajectory: swing up, then stay on the plan

The two designs above regulate about the upright, so they only work once the pole is already near it. Bringing it there from hanging is a **trajectory** problem: plan a swing-up $x_d(t), u_d(t)$ over $[0, t_f]$, then keep the real cart-pole on it. Around the plan the error $\tilde x = x - x_d(t)$ obeys a linear, time-varying model,
$$\dot{\tilde x} = A(t)\,\tilde x + B(t)\,\tilde u, \qquad A(t) = \frac{\partial f}{\partial x}\Big|_{x_d(t),\,u_d(t)},\quad B(t) = \frac{\partial f}{\partial u}\Big|_{x_d(t),\,u_d(t)},$$
which is exactly the setting of the finite-horizon LQR of section 5, with matrices that change along the trajectory. The plan itself comes from direct collocation: `TrajectoryOptimizationPlanner` turns the swing-up into a nonlinear program on $40$ knots over $4\,\mathrm{s}$, with the force limited to $\pm 10$ N.

In [ ]:
X_START = np.array([0.0, 0.0, 0.0, 0.0])  # hanging, at rest
OPTIMIZER = "ipopt" if importlib.util.find_spec("cyipopt") else "scipy_slsqp"

plant = CartPole()
plant.inputs["u"].lower_bound[0] = -10.0
plant.inputs["u"].upper_bound[0] = 10.0

swing_up = PlanningProblem(
    plant,
    tf=4.0,
    x_start=X_START,
    x_goal=X_BAR,
    cost=QuadraticCost.from_system(plant, Q=np.diag([1.0, 1.0, 0.0, 0.0]), R=np.diag([0.01]), xbar=X_BAR),
)
planner = TrajectoryOptimizationPlanner(
    swing_up, n_steps=40, transcription="direct_collocation", compile_backend="jax", optimizer_method=OPTIMIZER
)
reference = planner.solve().trajectory
planner.plot_solution()

`trajectory_lqr` linearizes the plant at every sample of the reference and sweeps the Riccati equation backward along it, one exact step per interval; the terminal weight defaults to the stationary solution at the end point, so past $t_f$ the block keeps balancing there. The result is the feedback
$$u = u_d(t) - K(t)\,\big(x - x_d(t)\big),$$
the planned input as feedforward plus a time-varying correction. The gains are small while the pole swings and large near the end, where the linearized dynamics are those of the upright.

In [ ]:
ctl_traj = trajectory_lqr(plant, reference, Q, R)
ctl_traj.plot_gain_schedule()

The real cart-pole starts half a metre away and tilted, not where the plan starts. Closed on the plant, the feedback pulls the state back onto the reference and holds the upright after $t_f$. `TrajectorySource` replays the same $u_d(t)$ with no feedback: the plan is feasible, but the pole falls.

In [ ]:
X0_SWING = X_START + np.array([0.5, 0.3, 0.0, 0.0])  # cart displaced, pole tilted

plant.x0 = X0_SWING
loop = ctl_traj @ plant
traj = loop.compute_trajectory(tf=8.0, verbose=False)
loop.plot_trajectory(traj, signals=("x", "u"))
loop.animate(traj)

replay = TrajectorySource(reference.t, reference.u) >> plant
traj_replay = replay.compute_trajectory(tf=8.0, verbose=False)
replay.plot_trajectory(traj_replay, signals=("x", "u"))

## 7. The backward recursion, by hand

The library propagated $-\dot S = S A + A^T S - S B R^{-1} B^T S + Q$ exactly, one matrix exponential per step. The textbook version is the discrete-time recursion on $S_k$, stepped backward from $S_N = S_f$; its continuous-time twin is one explicit Euler step per interval $\Delta t$:
$$S_{k-1} = S_k + \Delta t\,\big( S_k A + A^T S_k - S_k B R^{-1} B^T S_k + Q \big).$$
Six lines. `lqr_gain_schedule` returns the exact schedule as arrays $(t, K, S)$; the cell compares $S(0)$ from the recursion with it for several step sizes: accurate at small steps, finite but wrong at $\Delta t = 0.2\,\mathrm{s}$, and diverging just beyond.

In [ ]:
t, K_t, S_t = lqr_gain_schedule(A, B, Q, R, S_F, TF)  # the exact schedule, as arrays


def riccati_backward_euler(A, B, Q, R, S_f, tf, dt):
    """S(t) on a grid of step dt, from S(tf) = S_f backward, one Euler step at a time."""
    R_inv = np.linalg.inv(R)
    t = np.arange(0.0, tf + dt / 2, dt)
    S = np.zeros((len(t), A.shape[0], A.shape[0]))
    S[-1] = S_f
    for k in range(len(t) - 1, 0, -1):
        dS = S[k] @ A + A.T @ S[k] - S[k] @ B @ R_inv @ B.T @ S[k] + Q  # -dS/dt
        S[k - 1] = S[k] + dt * dS
    return t, S


with np.errstate(over="ignore", invalid="ignore"):  # let a diverging recursion overflow quietly
    for dt in (0.01, 0.1, 0.2, 0.25, 0.3):
        t_e, S_e = riccati_backward_euler(A, B, Q, R, S_F, TF, dt)
        err = np.max(np.abs(S_e[0] - S_t[0]))
        print(f"dt = {dt:5.2f}   max |S_euler(0) - S(0)| = {err:10.4g}   finite: {np.all(np.isfinite(S_e[0]))}")

## 8. Things to try

1. **Read the source.** Open [`control/lqr.py`](https://github.com/alx87grd/minilink/blob/main/minilink/control/lqr.py): `lqr_gain` solves the algebraic equation, `lqr_gain_schedule` propagates the differential one through the Hamiltonian matrix $H = \begin{bmatrix} A & -BR^{-1}B^T \\ -Q & -A^T \end{bmatrix}$. Find where $K = R^{-1} B^T S$ is formed in each, and why $S(t - \Delta t) = Y X^{-1}$ when $[X;\,Y] = e^{-H\Delta t}[I;\,S(t)]$.
2. **Basin of the constant gain.** Increase the initial tilt in `X0_LIST` until the infinite-horizon loop no longer recovers. Relate the failure to the force it requests and to the linear model's validity.
3. **Horizon and terminal weight.** Rerun section 5 with `TF = 3.0`, then with `S_F = 100 * np.eye(4)`. How does the schedule change near $t_f$, and what happens to the state at $t_f$?
4. **Past the horizon.** Simulate the finite-horizon loop with `tf=2 * TF`. By default the block holds its last gain: is that gain stabilizing? Look at $K(t_f)$ when $S_f = 0$, then rebuild the controller with `after=\"stationary\"` and compare.
5. **The Euler recursion.** Locate the step at which the recursion of section 7 diverges. The Riccati equation linearized about its solution evolves at the rates $\lambda_i + \lambda_j$ of the closed-loop poles (about $11\,\mathrm{rad/s}$ here), and an explicit Euler step is stable only while $|1 + \Delta t\, \lambda| < 1$: check that the threshold matches, and whether $S$ stays symmetric and positive definite on the way to it.
6. **Replay from the planned start.** Set `plant.x0 = X_START` and run the replay of section 6 again. It still falls: collocation enforces the dynamics at the knots only, the simulator integrates between them, and the upright amplifies every mismatch. Where along the trajectory does the divergence begin?
7. **Tracking weights.** Redesign `ctl_traj` with `Q = np.diag([10, 10, 1, 1])`, then with `R = np.diag([10])`. Read the change on the gain schedule and on the peak force of the closed loop, and say which one the $\pm 10$ N limit of the plan cares about.
8. **Terminal weight.** Pass `S_f=Q` to `trajectory_lqr` instead of the default. What does the block do after $t_f$, and why is the stationary solution the safer default when the trajectory ends at an equilibrium?